In [1]:
# Runtime evidence: this cell checks the environment actually used by the kernel.
import sys as _shape_sys, json as _shape_json, importlib.util as _shape_util
print("SHAPE_RUNTIME_PROVENANCE=" + _shape_json.dumps({
    "python_executable": _shape_sys.executable,
    "python_version": _shape_sys.version.split()[0],
    "shape_lab_origin": _shape_util.find_spec("shape_lab").origin
}))
del _shape_sys, _shape_json, _shape_util


SHAPE_RUNTIME_PROVENANCE={"python_executable": "/opt/pyvenv/bin/python", "python_version": "3.13.5", "shape_lab_origin": "/tmp/shape-course-q90rdcpz/course/src/shape_lab/__init__.py"}


# Stage 03 — Reference coding solutions

These are instructor reference implementations, not evidence of your independent mastery. Run after attempting your own version. All examples use explicit small inputs and the declared dataset partitions.

In [2]:
from IPython import get_ipython
if get_ipython() is not None:
    get_ipython().run_line_magic('matplotlib','inline')
from pathlib import Path
import sys, json
from itertools import combinations
import numpy as np
import pandas as pd
ROOT=next((p for p in [Path.cwd(),*Path.cwd().parents] if (p/'data/manifest.json').exists() and (p/'src/shape_lab').exists()),None)
if ROOT is None: raise RuntimeError('Open this notebook inside the extracted course folder.')
sys.path.insert(0,str(ROOT/'src'))
from shape_lab.data import load_iris_data,load_digit_data,digit_example
from shape_lab.algebra import closure,validate_complex,rank_mod2,boundary_matrix,betti_numbers
from shape_lab.topology import component_labels
from shape_lab.persistence import ordered_filtration,rips_filtration,reduce_boundary,persistent_homology,diagram,pixel_filtration,finite_bottleneck
print('Working inside',ROOT.name,'— no network data downloads.')

Working inside course — no network data downloads.


## 03.C1 — Pull back a set through a map

Given a total mapping dictionary and a codomain subset V, return its inverse image.

**Before running:** describe the input type, output type, and a hand-checkable expected result.

<details><summary>Hint</summary>

Iterate over dictionary input-output pairs.

</details>

In [3]:
def pullback(mapping,V):
    return {x for x,y in mapping.items() if y in V}

In [4]:
assert pullback({'a':0,'b':1,'c':1},{1})=={'b','c'}
assert pullback({'a':0},set())==set()
print("Inverse images computed")

Inverse images computed


**Why this works and what it does not establish:** The operation works even when several inputs share an output. It is exactly the operation appearing in the topological definition of continuity.

## 03.C2 — Check continuity on finite spaces

Assume the supplied open-set collections are valid topologies and mapping is total. Return whether every codomain open set has an open inverse image.

**Before running:** describe the input type, output type, and a hand-checkable expected result.

<details><summary>Hint</summary>

Do not test only points; test all declared open subsets of the codomain.

</details>

In [5]:
def continuous_on_finite(mapping,domain_opens,codomain_opens):
    T=set(map(frozenset,domain_opens))
    return all(frozenset(x for x,y in mapping.items() if y in V) in T for V in codomain_opens)

In [6]:
discrete=[set(),{0},{1},{0,1}];indiscrete=[set(),{0,1}];f={0:0,1:1}
assert continuous_on_finite(f,discrete,indiscrete)
assert not continuous_on_finite(f,indiscrete,discrete)
print("Identity continuous in one direction, not the reverse")

Identity continuous in one direction, not the reverse


**Why this works and what it does not establish:** The identity is a bijection of the underlying sets, yet changing the topologies can make its inverse discontinuous. This is a concrete reason that continuous bijection is not the definition of homeomorphism.

## 03.C3 — Construct an inverse when it exists

Return the inverse dictionary of a finite injective mapping. Raise ValueError when two inputs share an output.

**Before running:** describe the input type, output type, and a hand-checkable expected result.

<details><summary>Hint</summary>

Compare the number of unique outputs with the number of inputs.

</details>

In [7]:
def inverse_mapping(mapping):
    if len(set(mapping.values()))!=len(mapping):
        raise ValueError('The map is not injective.')
    return {y:x for x,y in mapping.items()}

In [8]:
assert inverse_mapping({'a':2,'b':7})=={2:'a',7:'b'}
try: inverse_mapping({'a':1,'b':1})
except ValueError: pass
else: raise AssertionError('Non-injective map accepted')
print("Inverse and non-injective case checked")

Inverse and non-injective case checked


**Why this works and what it does not establish:** This constructs an inverse onto the image. Surjectivity onto a separately declared larger codomain remains a different condition; dictionary inversion alone does not prove it.

## 03.C4 — Count graph components

Count connected components of a symmetric boolean adjacency matrix using a search. An isolated vertex is its own component.

**Before running:** describe the input type, output type, and a hand-checkable expected result.

<details><summary>Hint</summary>

Start a new search only from an unvisited vertex.

</details>

In [9]:
def graph_component_count(A):
    A=np.asarray(A,dtype=bool);seen=set();count=0
    for start in range(len(A)):
        if start in seen: continue
        count+=1;stack=[start];seen.add(start)
        while stack:
            v=stack.pop()
            for w in np.flatnonzero(A[v]):
                if int(w) not in seen:
                    seen.add(int(w));stack.append(int(w))
    return count

In [10]:
assert graph_component_count([[0,1,0],[1,0,0],[0,0,0]])==2
X=load_iris_data().iloc[:12,1:5].to_numpy()
D=np.linalg.norm(X[:,None,:]-X[None,:,:],axis=2)
A=D<=.3
assert graph_component_count(A)==len(np.unique(component_labels(A)))
print("Real-data threshold graph components:",graph_component_count(A))

Real-data threshold graph components: 5


**Why this works and what it does not establish:** This counts components of the chosen finite graph, not plant species. Different thresholds give different graphs. For the geometric realization of a finite graph, connected and path-connected components agree.

## 03.C5 — Measure information lost in a projection

For a feature table X and distinct rows i,j, return (full distance, distance after retaining the first two columns).

**Before running:** describe the input type, output type, and a hand-checkable expected result.

<details><summary>Hint</summary>

A coordinate projection discards some squared terms from the Euclidean distance.

</details>

In [11]:
def projection_distances(X,i,j):
    delta=np.asarray(X,dtype=float)[i]-np.asarray(X,dtype=float)[j]
    return float(np.linalg.norm(delta)),float(np.linalg.norm(delta[:2]))

In [12]:
assert projection_distances([[0,0,0],[0,0,5]],0,1)==(5.,0.)
X=load_iris_data().iloc[:,1:5].to_numpy()
full,projected=projection_distances(X,0,75)
assert projected<=full+1e-12
print("Measured pair full/projected distances:",full,projected)

Measured pair full/projected distances: 3.59722114972099 1.5811388300841898


**Why this works and what it does not establish:** A continuous projection can identify points that were distinct. A two-dimensional display is therefore not a certificate that the original higher-dimensional dataset has the same topology.